# Tree Of Thought

#### Note: ToT is very expensive and only used in research


## The ToT Loop: Propose, Evaluate, Search
A ToT system consists of three modules:

- Thought Proposer: Generates 3-5 potential "next steps" for a problem.
- State Evaluator: Grades each step (e.g., "Good", "Maybe", "Impossible").
- Search Algorithm: (BFS or DFS) to decide which branch to explore next.

Production Reality (2026)

Pure ToT is rarely deployed exactly as in the research paper because it is expensive.

A branching factor of 5 and depth of 4 means:

5^4 = 625

states to evaluate.

That means hundreds of LLM calls.

Instead, modern agent systems often use:

Plan
 ↓
Generate 3 candidates
 ↓
Score candidates
 ↓
Keep top 1-2
 ↓
Continue

This is essentially a beam search version of ToT.

In [ ]:
import os
import json
import re
from dataclasses import dataclass
from typing import List

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate


# -----------------------------
# CONFIG
# -----------------------------
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

BEAM_WIDTH = 2        # keep top 2 branches
BRANCH_FACTOR = 3     # generate 3 candidates per branch
MAX_DEPTH = 4        # how many rounds to expand


# -----------------------------
# HELPERS
# -----------------------------
def extract_json(text: str) -> dict:
    """
    Extract JSON object from model output.
    Assumes the model returns a single JSON object.
    """
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in: {text}")
    return json.loads(match.group(0))


def format_path(path: List[str]) -> str:
    if not path:
        return "(empty)"
    return "\n".join(f"{i+1}. {step}" for i, step in enumerate(path))


@dataclass
class BeamState:
    steps: List[str]
    total_score: float = 0.0

    @property
    def avg_score(self) -> float:
        if not self.steps:
            return 0.0
        return self.total_score / len(self.steps)

    @property
    def text(self) -> str:
        return format_path(self.steps)


# -----------------------------
# PROMPTS
# -----------------------------
propose_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a careful reasoning assistant. "
        "Generate candidate next steps for solving the problem. "
        "Return ONLY valid JSON."
    ),
    (
        "human",
        """
Problem:
{problem}

Current reasoning path:
{path}

Generate exactly {n} distinct possible next reasoning steps.
Each step should move the solution forward.

Return ONLY this JSON:
{{
  "candidates": [
    "candidate 1",
    "candidate 2",
    "candidate 3"
  ]
}}
"""
    )
])

score_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a strict evaluator. "
        "Score the candidate step from 1 to 10 based on usefulness, correctness, and progress toward the answer. "
        "Return ONLY valid JSON."
    ),
    (
        "human",
        """
Problem:
{problem}

Current reasoning path:
{path}

Candidate next step:
{candidate}

Return ONLY this JSON:
{{
  "score": 1,
  "reason": "brief reason"
}}
"""
    )
])

judge_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a verifier. Decide whether the current reasoning path is already enough to answer the problem correctly. "
        "Return ONLY valid JSON."
    ),
    (
        "human",
        """
Problem:
{problem}

Reasoning path:
{path}

Is this path complete enough to produce the final answer?
Return ONLY this JSON:
{{
  "solved": true,
  "reason": "brief reason"
}}
"""
    )
])

final_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a final answer generator. Use the best reasoning path to produce the final answer clearly and concisely."
    ),
    (
        "human",
        """
Problem:
{problem}

Best reasoning path:
{path}

Now give the final answer only. If useful, include one short explanation.
"""
    )
])

propose_chain = propose_prompt | llm
score_chain = score_prompt | llm
judge_chain = judge_prompt | llm
final_chain = final_prompt | llm


# -----------------------------
# LLM CALLS
# -----------------------------
def propose_steps(problem: str, state: BeamState, n: int = 3) -> List[str]:
    response = propose_chain.invoke({
        "problem": problem,
        "path": state.text,
        "n": n
    })
    data = extract_json(response.content)
    candidates = data.get("candidates", [])
    return [c.strip() for c in candidates if c.strip()]


def score_step(problem: str, state: BeamState, candidate: str) -> int:
    response = score_chain.invoke({
        "problem": problem,
        "path": state.text,
        "candidate": candidate
    })
    data = extract_json(response.content)
    score = data.get("score", 1)

    try:
        score = int(score)
    except Exception:
        score = 1

    return max(1, min(10, score))


def is_solved(problem: str, state: BeamState) -> bool:
    response = judge_chain.invoke({
        "problem": problem,
        "path": state.text
    })
    data = extract_json(response.content)
    return bool(data.get("solved", False))


def get_final_answer(problem: str, state: BeamState) -> str:
    response = final_chain.invoke({
        "problem": problem,
        "path": state.text
    })
    return response.content


# -----------------------------
# BEAM SEARCH ToT
# -----------------------------
def beam_search_tot(problem: str,
                    beam_width: int = 2,
                    branch_factor: int = 3,
                    max_depth: int = 4,
                    verbose: bool = True) -> str:
    beam = [BeamState(steps=[], total_score=0.0)]
    best_solved_state = None

    for depth in range(max_depth):
        if verbose:
            print(f"\n=== DEPTH {depth + 1} ===")

        expanded: List[BeamState] = []

        for idx, state in enumerate(beam):
            if verbose:
                print(f"\nBranch {idx + 1} current path:")
                print(state.text or "(empty)")
                print(f"Avg score: {state.avg_score:.2f}")

            try:
                candidates = propose_steps(problem, state, n=branch_factor)
            except Exception as e:
                if verbose:
                    print(f"Proposal failed: {e}")
                continue

            if verbose:
                print("\nCandidates:")
                for c in candidates:
                    print(f"- {c}")

            for cand in candidates:
                try:
                    cand_score = score_step(problem, state, cand)
                except Exception as e:
                    if verbose:
                        print(f"Scoring failed for '{cand}': {e}")
                    cand_score = 1

                new_steps = state.steps + [cand]
                new_total = state.total_score + cand_score
                new_state = BeamState(steps=new_steps, total_score=new_total)
                expanded.append(new_state)

                if verbose:
                    print(f"  Score: {cand_score}")

        if not expanded:
            break

        expanded.sort(key=lambda s: s.avg_score, reverse=True)
        beam = expanded[:beam_width]

        if verbose:
            print("\nTop beam states:")
            for i, state in enumerate(beam, start=1):
                print(f"\n--- TOP {i} ---")
                print(state.text)
                print(f"Avg score: {state.avg_score:.2f}")

        for state in beam:
            try:
                if is_solved(problem, state):
                    best_solved_state = state
                    if verbose:
                        print("\nA branch is judged solved early.")
                    break
            except Exception:
                pass

        if best_solved_state is not None:
            break

    final_state = best_solved_state if best_solved_state is not None else beam[0]
    return get_final_answer(problem, final_state)


# -----------------------------
# EXAMPLE
# -----------------------------
if __name__ == "__main__":
    problem = "If a train travels 120 km in 2 hours, what is its speed?"
    answer = beam_search_tot(
        problem,
        beam_width=BEAM_WIDTH,
        branch_factor=BRANCH_FACTOR,
        max_depth=MAX_DEPTH,
        verbose=True
    )

    print("\n===== FINAL ANSWER =====")
    print(answer)

In the code I gave, it evaluates every candidate step at every level.

Suppose the problem is:

Find the speed of a train.
Depth 1

The Thought Proposer generates:

A: Identify distance
B: Identify time
C: Use speed formula

The Evaluator scores all three:

A → 8
B → 7
C → 9

If beam_width = 2, we keep:

C (9)
A (8)

and discard:

B (7)
Depth 2

Now we expand BOTH surviving branches.

Branch C

Current path:

Use speed formula

Generate:

C1: Speed = Distance / Time
C2: Gather values
C3: Convert units

Score:

C1 → 9
C2 → 8
C3 → 3
Branch A

Current path:

Identify distance

Generate:

A1: Distance = 120 km
A2: Check units
A3: Estimate speed

Score:

A1 → 10
A2 → 6
A3 → 2

Now the search sees:

C1 → 9
C2 → 8
A1 → 10
A2 → 6

and again keeps the top branches.

So what exactly is evaluated?

The evaluator evaluates every newly generated thought.

State
   ↓
Generate 3 thoughts
   ↓
Score thought 1
Score thought 2
Score thought 3
   ↓
Keep best

Then:

Best thought
   ↓
Generate new thoughts
   ↓
Score all new thoughts
Difference from Chain of Thought
CoT
Step 1
  ↓
Step 2
  ↓
Step 3

No evaluation.

If Step 1 is bad:

Bad Step 1
   ↓
Bad Step 2
   ↓
Bad Answer
ToT
Step 1A
Step 1B
Step 1C
    ↓
Evaluate
    ↓
Keep best

The system can reject bad reasoning early.

What does the evaluator actually look at?

Typically:

Problem
Current State
Candidate Thought

Example:

Problem:
Train traveled 120 km in 2 hours.

Current State:
Distance identified.

Candidate:
Speed = Distance / Time

The evaluator asks:

Is this thought:
- Correct?
- Useful?
- Bringing us closer to a solution?

and returns:

{
  "score": 9
}
Production Systems

Modern agent systems usually don't score every step forever because it's expensive.

A common production approach is:

Generate 5 candidates
      ↓
Score 5 candidates
      ↓
Keep top 2
      ↓
Generate again

This is called Beam Search.

For example:

Depth 1: 5 thoughts
Depth 2: 2 × 5 = 10 thoughts
Depth 3: 2 × 5 = 10 thoughts

Total evaluated:

5 + 10 + 10 = 25 thoughts

instead of:

5³ = 125 thoughts

which a full Tree-of-Thoughts search would evaluate.

That's why most real-world AI agents use beam-search ToT, not the original paper's exhaustive BFS search.